[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/onnx/tutorials/blob/main/05_ONNX_Operators_and_OpSets/04_Operator_Schemas/Operator_Schemas_Deep_Dive.ipynb)

# 5.4 Operator Schemas — Deep Dive

## Table of Contents

| # | Section | Description |
|---|---------|-------------|
| 1 | [What is an Operator Schema?](#1-what-is-an-operator-schema) | The legal contract for every ONNX op |
| 2 | [Schema Components](#2-schema-components) | FormalParameter, attributes, type constraints, doc |
| 3 | [Type Constraints](#3-type-constraints) | Formal type system with type variables |
| 4 | [Broadcasting Rules](#4-broadcasting-rules) | Numpy-style broadcasting with formal definitions |
| 5 | [Querying Schemas Programmatically](#5-querying-schemas) | Using onnx.defs for schema inspection |
| 6 | [Reading Operator Documentation](#6-reading-documentation) | Structured approach to understanding ops |
| 7 | [Attribute Versioning](#7-attribute-versioning) | How attributes change across opset versions |
| 8 | [Shape Inference](#8-shape-inference) | Formal shape propagation rules |
| 9 | [Complete Schema Audit](#9-complete-schema-audit) | Auditing all schemas in a model |
| 10 | [Key Takeaways](#10-key-takeaways) | Summary |

In [ ]:
# !pip install onnx numpy matplotlib --quiet

import numpy as np
import onnx
from onnx import helper, TensorProto, checker, numpy_helper, defs
from onnx import shape_inference
import textwrap

opset = defs.onnx_opset_version()
print(f"ONNX version: {onnx.__version__}")
print(f"Default opset: {opset}")

<a id='1-what-is-an-operator-schema'></a>
## 1. What is an Operator Schema?

An operator schema is the **authoritative contract** that defines exactly what an
operator does: its inputs, outputs, attributes, permitted types, shape relationships,
and semantic behavior. Think of it as the "legal document" governing node execution.

### The Schema as a Contract

```
┌──────────────────────────────────────────────────────────────────────┐
│                     OPERATOR SCHEMA CONTRACT                        │
│                    (onnx.defs.OpSchema)                              │
├──────────────────────────────────────────────────────────────────────┤
│                                                                      │
│  ┌─────────────────────┐     ┌──────────────────────────────────┐   │
│  │ IDENTITY             │     │ FORMAL PARAMETERS                │   │
│  │  name: "Conv"        │     │  inputs:                         │   │
│  │  domain: ""          │     │    X: T (Single)                 │   │
│  │  since_version: 11   │     │    W: T (Single)                 │   │
│  └─────────────────────┘     │    B: T (Optional)               │   │
│                               │  outputs:                        │   │
│  ┌─────────────────────┐     │    Y: T (Single)                 │   │
│  │ TYPE CONSTRAINTS     │     └──────────────────────────────────┘   │
│  │  T ∈ {float16,      │                                            │
│  │       float32,       │     ┌──────────────────────────────────┐   │
│  │       float64}       │     │ ATTRIBUTES                       │   │
│  └─────────────────────┘     │  kernel_shape: ints (required)   │   │
│                               │  strides: ints (default=[1,1])   │   │
│  ┌─────────────────────┐     │  pads: ints (default=[0,...])     │   │
│  │ DOCUMENTATION        │     │  dilations: ints (default=[1,1]) │   │
│  │  Semantic description│     │  group: int (default=1)          │   │
│  │  Examples            │     │  auto_pad: string (NOTSET)       │   │
│  │  References          │     └──────────────────────────────────┘   │
│  └─────────────────────┘                                            │
│                                                                      │
│  ★ If a node violates this contract, the model is INVALID            │
│  ★ Runtimes MUST implement this contract faithfully                  │
└──────────────────────────────────────────────────────────────────────┘
```

### Schema vs Node

| Concept | Schema (`OpSchema`) | Node (`NodeProto`) |
|---------|--------------------|-----------------------|
| Role | **Definition** (type-level) | **Instance** (value-level) |
| Contains | Type constraints, arity rules | Actual tensor names, attribute values |
| Scope | Shared across all uses | Specific to one graph position |
| Analogy | Function signature | Function call |

In [ ]:
# Inspect the schema object structure
def print_schema_full(op_type, opset_ver=opset, domain=""):
    """Print a comprehensive schema report."""
    s = defs.get_schema(op_type, opset_ver, domain)

    print(f"{'=' * 70}")
    print(f"  {s.name}  (domain={domain!r}, since_version={s.since_version})")
    print(f"{'=' * 70}")

    print(f"\n  INPUTS ({len(s.inputs)}):")
    for p in s.inputs:
        opt = getattr(p, 'option', 'Single')
        print(f"    {p.name:<15} type={p.type_str:<5} arity={opt}")

    print(f"\n  OUTPUTS ({len(s.outputs)}):")
    for p in s.outputs:
        opt = getattr(p, 'option', 'Single')
        print(f"    {p.name:<15} type={p.type_str:<5} arity={opt}")

    print(f"\n  TYPE CONSTRAINTS ({len(s.type_constraints)}):")
    for tc in s.type_constraints:
        types = list(tc.allowed_type_strs)
        preview = ', '.join(types[:6])
        suffix = f' ... (+{len(types)-6})' if len(types) > 6 else ''
        print(f"    {tc.type_param_str:<5} ∈ {{{preview}{suffix}}}")

    print(f"\n  ATTRIBUTES ({len(s.attributes)}):")
    for name, attr in sorted(s.attributes.items()):
        req = 'required' if attr.required else 'optional'
        print(f"    {name:<20} type={attr.type:<15} {req}")

    doc = s.doc.strip() if s.doc else "(no doc)"
    doc_preview = doc[:150].replace('\n', ' ')
    print(f"\n  DOC: {doc_preview}...")
    print()

# Inspect Conv schema
print_schema_full("Conv")

In [ ]:
# Compare schemas of related operators
for op in ["MatMul", "Gemm", "Softmax", "Relu"]:
    print_schema_full(op)

<a id='2-schema-components'></a>
## 2. Schema Components

Every operator schema consists of several formal components:

### 2.1 Formal Parameters (Inputs/Outputs)

Each input or output is a `FormalParameter` with:

| Property | Description | Example |
|----------|-------------|----------|
| `name` | Parameter identifier | `"X"`, `"W"`, `"B"` |
| `type_str` | Type variable name | `"T"`, `"T1"`, `"I"` |
| `option` | Arity constraint | Single, Optional, Variadic |
| `description` | Human-readable docs | "Input data tensor" |

### 2.2 Arity Rules

```
  Single:    ──────────[ op ]──────────    Exactly 1 tensor
  Optional:  ─ ─ ─ ─ ─[ op ]──────────    0 or 1 tensors
  Variadic:  ══════════[ op ]══════════    1+ tensors (e.g., Concat inputs)
```

### 2.3 Attributes

Attributes are **static** configuration embedded in the graph definition:

| Attribute Type | ONNX Enum | Example |
|---------------|-----------|----------|
| `INT` | `AttributeProto.INT` | `axis=1` |
| `FLOAT` | `AttributeProto.FLOAT` | `alpha=0.01` |
| `STRING` | `AttributeProto.STRING` | `auto_pad="SAME"` |
| `TENSOR` | `AttributeProto.TENSOR` | `value=Constant(...)` |
| `INTS` | `AttributeProto.INTS` | `kernel_shape=[3,3]` |
| `FLOATS` | `AttributeProto.FLOATS` | `scales=[0.5, 0.5]` |
| `GRAPH` | `AttributeProto.GRAPH` | `body=SubGraph(...)` |

In [ ]:
# Enumerate all attribute types used across operators
from collections import Counter

attr_type_names = {
    1: 'FLOAT', 2: 'INT', 3: 'STRING', 4: 'TENSOR', 5: 'GRAPH',
    6: 'FLOATS', 7: 'INTS', 8: 'STRINGS', 9: 'TENSORS', 10: 'GRAPHS',
    11: 'SPARSE_TENSOR', 12: 'SPARSE_TENSORS', 13: 'TYPE_PROTO', 14: 'TYPE_PROTOS',
}

attr_type_counts = Counter()
attr_examples = {}

for schema in defs.get_all_schemas_with_history():
    if schema.domain not in ('', 'ai.onnx'):
        continue
    for attr_name, attr in schema.attributes.items():
        type_name = attr_type_names.get(attr.type, f'UNKNOWN({attr.type})')
        attr_type_counts[type_name] += 1
        if type_name not in attr_examples:
            attr_examples[type_name] = f"{schema.name}.{attr_name}"

print(f"{'Attribute Type':<20} │ {'Count':>6} │ Example")
print("─" * 60)
for type_name, count in attr_type_counts.most_common():
    example = attr_examples.get(type_name, '')
    print(f"{type_name:<20} │ {count:>6} │ {example}")

# Show input arity distribution
arity_counts = {'Single': 0, 'Optional': 0, 'Variadic': 0}
latest_schemas = {}
for schema in defs.get_all_schemas_with_history():
    if schema.domain in ('', 'ai.onnx'):
        key = schema.name
        if key not in latest_schemas or schema.since_version > latest_schemas[key].since_version:
            latest_schemas[key] = schema

for schema in latest_schemas.values():
    for p in schema.inputs:
        opt = str(getattr(p, 'option', 'Single'))
        for arity in arity_counts:
            if arity.lower() in opt.lower():
                arity_counts[arity] += 1
                break

print(f"\nInput parameter arity distribution:")
for arity, count in arity_counts.items():
    bar = '█' * (count // 3)
    print(f"  {arity:<12} {count:>4}  {bar}")

<a id='3-type-constraints'></a>
## 3. Type Constraints

Type constraints form the **type system** of ONNX operators. They use **type variables**
(like generics in programming languages) to enforce type consistency.

### Formal Definition

A type constraint is a pair $(T_v, \mathcal{S}_v)$ where:
- $T_v$ is a **type variable** (e.g., `T`, `T1`, `I`)
- $\mathcal{S}_v$ is the **allowed type set** (e.g., $\{\text{float16}, \text{float32}, \text{float64}\}$)

### Type Binding Rules

When a graph is validated, type variables are **bound** to concrete types:

$$\text{bind}(T_v) = \text{elem\_type}(\text{first input using } T_v)$$

All subsequent inputs/outputs using the same type variable must have the **same** element type:

$$\forall \; p_i, p_j \text{ using } T_v: \; \text{elem\_type}(p_i) = \text{elem\_type}(p_j)$$

### Type Constraint Flow

```
  Type Constraint: T ∈ {float16, float32, float64}

  Input X: float32  ───┐
                        │
                   bind(T) = float32
                        │
  Input W: float32  ───┤   ✓  same T → OK
                        │
  Input B: float32  ───┤   ✓  same T → OK
                        │
  Output Y: float32 ◀──┘   ✓  T propagates to output


  ERROR CASE:
  Input X: float32  ───┐
                        │  bind(T) = float32
  Input W: float16  ───┤  ✗  CONFLICT! float16 ≠ float32
                        │     → model validation fails
```

### Multiple Type Variables

Some operators use multiple independent type variables:

$$T_1 \in \{\text{float16}, \text{float32}, \text{float64}\} \quad \text{(data)}$$
$$T_2 \in \{\text{int32}, \text{int64}\} \quad \text{(indices)}$$

For example, `Gather`:
- Input `data`: type $T$ (can be any numeric type)
- Input `indices`: type $T_{ind}$ (must be int32 or int64)
- Output: type $T$ (same as data)

The two type variables bind independently — data can be float32 while indices are int64.

In [ ]:
# Analyze type constraints across all operators
def get_type_constraints(op_type, opset_ver=opset, domain=""):
    """Extract type constraints for an operator."""
    try:
        schema = defs.get_schema(op_type, opset_ver, domain)
        constraints = {}
        for tc in schema.type_constraints:
            constraints[tc.type_param_str] = list(tc.allowed_type_strs)
        return constraints
    except Exception:
        return {}

# Compare type constraints across operators
ops_to_check = ["Add", "MatMul", "Conv", "Relu", "Cast", "Gather",
                "Where", "ReduceSum", "Softmax", "Reshape"]

print(f"{'Operator':<15} │ {'Type Vars':>10} │ {'Types per Var':>15} │ Details")
print("─" * 80)
for op in ops_to_check:
    tc = get_type_constraints(op)
    if tc:
        num_vars = len(tc)
        details = []
        for var, types in tc.items():
            preview = ', '.join(t.replace('tensor(', '').rstrip(')') for t in types[:4])
            suffix = f'+{len(types)-4}' if len(types) > 4 else ''
            details.append(f"{var}:{{{preview}{suffix}}}")
        types_per = ', '.join(str(len(v)) for v in tc.values())
        detail_str = '  '.join(details)
        print(f"{op:<15} │ {num_vars:>10} │ {types_per:>15} │ {detail_str}")

# Demonstrate type validation
print(f"\nType validation examples:")
for dtype_x, dtype_w, expected in [
    (TensorProto.FLOAT, TensorProto.FLOAT, "valid"),
    (TensorProto.FLOAT, TensorProto.DOUBLE, "invalid (type mismatch)"),
    (TensorProto.DOUBLE, TensorProto.DOUBLE, "valid"),
]:
    X = helper.make_tensor_value_info("X", dtype_x, [2, 3])
    W = helper.make_tensor_value_info("W", dtype_w, [3, 4])
    Y = helper.make_tensor_value_info("Y", dtype_x, [2, 4])
    node = helper.make_node("MatMul", ["X", "W"], ["Y"])
    graph = helper.make_graph([node], "type_test", [X, W], [Y])
    model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])
    try:
        checker.check_model(model)
        status = "VALID"
    except Exception as e:
        status = f"REJECTED"
    x_name = TensorProto.DataType.Name(dtype_x)
    w_name = TensorProto.DataType.Name(dtype_w)
    print(f"  MatMul(X:{x_name}, W:{w_name}): {status} (expected: {expected})")

<a id='4-broadcasting-rules'></a>
## 4. Broadcasting Rules

Broadcasting determines how operators handle inputs with different shapes.
ONNX supports two broadcasting modes defined in operator schemas.

### Numpy-Style (Multidirectional) Broadcasting

Two dimensions are **compatible** if:

$$\text{compat}(d_A, d_B) \iff (d_A = d_B) \lor (d_A = 1) \lor (d_B = 1)$$

The output dimension is:

$$d_{\text{out}} = \max(d_A, d_B)$$

Shapes are **right-aligned**: if ranks differ, the shorter shape is padded with 1s on the left.

### Broadcasting Algorithm

```
  broadcast(shape_A, shape_B):
    1. Determine max rank: R = max(rank(A), rank(B))
    2. Pad shorter shape with leading 1s:
       A_padded = [1, ..., 1, a_1, a_2, ..., a_m]
       B_padded = [1, ..., 1, b_1, b_2, ..., b_n]
    3. For each dimension i from 0 to R-1:
       If A_padded[i] == B_padded[i]:  out[i] = A_padded[i]
       Elif A_padded[i] == 1:          out[i] = B_padded[i]
       Elif B_padded[i] == 1:          out[i] = A_padded[i]
       Else: ERROR (incompatible)

  Examples:
    [3, 4] ⊕ [4]        → [3, 4]       (bias addition)
    [3, 1] ⊕ [1, 4]     → [3, 4]       (outer product)
    [8, 1, 6, 1] ⊕ [7, 1, 5] → [8, 7, 6, 5]
    [3, 4] ⊕ [5, 4]     → ERROR        (3 ≠ 5, neither is 1)
```

### Multidirectional vs Unidirectional Broadcasting

| Type | Rule | Example Op | Description |
|------|------|-----------|-------------|
| **Multidirectional** | Both operands stretch | Add, Mul, Sub, Div | Standard numpy |
| **Unidirectional** | Only B stretches to A | Gemm (bias C) | B must be broadcastable to A |

### Shape Inference Formula for Broadcasting

Given inputs $A \in \mathbb{R}^{d_{a,1} \times \ldots \times d_{a,m}}$ and
$B \in \mathbb{R}^{d_{b,1} \times \ldots \times d_{b,n}}$:

$$\text{shape}(A \oplus B) = \left[\max(\tilde{d}_{a,i}, \tilde{d}_{b,i})\right]_{i=1}^{\max(m,n)}$$

where $\tilde{d}$ denotes the right-aligned, 1-padded dimensions.

In [ ]:
# Comprehensive broadcasting analysis
def broadcast_shapes(shape_a, shape_b):
    """Compute broadcast shape using ONNX/numpy rules."""
    rank = max(len(shape_a), len(shape_b))
    a = [1] * (rank - len(shape_a)) + list(shape_a)
    b = [1] * (rank - len(shape_b)) + list(shape_b)
    result = []
    for da, db in zip(a, b):
        if da == db:
            result.append(da)
        elif da == 1:
            result.append(db)
        elif db == 1:
            result.append(da)
        else:
            raise ValueError(f"Incompatible: {da} vs {db}")
    return tuple(result)

# Test cases organized by category
test_cases = {
    "Scalar broadcast": [
        ((3, 4), (1,)),
        ((1,), (5, 3)),
    ],
    "Bias addition": [
        ((3, 4), (4,)),
        ((2, 3, 4), (4,)),
        ((8, 16), (16,)),
    ],
    "Outer product": [
        ((3, 1), (1, 4)),
        ((5, 1, 1), (1, 3, 1)),
    ],
    "Complex": [
        ((8, 1, 6, 1), (7, 1, 5)),
        ((1, 1, 1, 1), (2, 3, 4, 5)),
        ((256, 1), (1, 256)),
    ],
}

for category, cases in test_cases.items():
    print(f"\n{category}:")
    for sa, sb in cases:
        try:
            result = broadcast_shapes(sa, sb)
            print(f"  {str(sa):>20} ⊕ {str(sb):<20} → {result}")
        except ValueError as e:
            print(f"  {str(sa):>20} ⊕ {str(sb):<20} → ERROR: {e}")

# Incompatible cases
print(f"\nIncompatible (should error):")
bad_cases = [((3, 4), (5, 4)), ((2, 3), (4, 5)), ((3,), (4,))]
for sa, sb in bad_cases:
    try:
        result = broadcast_shapes(sa, sb)
        print(f"  {str(sa):>20} ⊕ {str(sb):<20} → {result} (unexpected!)")
    except ValueError:
        print(f"  {str(sa):>20} ⊕ {str(sb):<20} → ERROR (expected)")

In [ ]:
# Verify broadcasting with actual ONNX models
import matplotlib.pyplot as plt

def verify_broadcast_add(shape_a, shape_b):
    """Build and run an Add model to verify broadcasting."""
    A = helper.make_tensor_value_info("A", TensorProto.FLOAT, list(shape_a))
    B = helper.make_tensor_value_info("B", TensorProto.FLOAT, list(shape_b))
    C = helper.make_tensor_value_info("C", TensorProto.FLOAT, None)
    model = helper.make_model(
        helper.make_graph(
            [helper.make_node("Add", ["A", "B"], ["C"])],
            "bc_test", [A, B], [C]),
        opset_imports=[helper.make_opsetid("", 17)])
    model = shape_inference.infer_shapes(model)
    
    inferred = [d.dim_value for d in model.graph.output[0].type.tensor_type.shape.dim]
    
    try:
        from onnx.reference import ReferenceEvaluator
        ev = ReferenceEvaluator(model)
        a = np.ones(shape_a, dtype=np.float32)
        b = np.ones(shape_b, dtype=np.float32) * 2
        c = ev.run(None, {"A": a, "B": b})[0]
        return tuple(inferred), c.shape, True
    except Exception:
        return tuple(inferred), None, False

print(f"{'Shape A':>15} {'Shape B':>15} │ {'Inferred':>15} {'Actual':>15} │ Match")
print("─" * 75)
for sa, sb in [((3,4),(4,)), ((2,3,4),(1,3,1)), ((8,1,6,1),(7,1,5)),
               ((1,),(5,3)), ((256,1),(1,256))]:
    inferred, actual, ok = verify_broadcast_add(sa, sb)
    match = "✓" if ok and inferred == actual else "✗"
    print(f"{str(sa):>15} {str(sb):>15} │ {str(inferred):>15} {str(actual):>15} │ {match}")

<a id='5-querying-schemas'></a>
## 5. Querying Schemas Programmatically

The `onnx.defs` module provides comprehensive APIs for schema inspection:

### Key API Functions

| Function | Purpose |
|----------|----------|
| `defs.get_schema(op, opset, domain)` | Get specific schema |
| `defs.get_all_schemas()` | All schemas at latest version |
| `defs.get_all_schemas_with_history()` | All schemas across all versions |
| `defs.onnx_opset_version()` | Current default opset version |
| `defs.has(op, opset)` | Check if op exists at opset |

### OpSchema Properties

| Property | Type | Description |
|----------|------|-------------|
| `name` | str | Operator name |
| `domain` | str | Domain ("" for default) |
| `since_version` | int | OpSet version introduced |
| `inputs` | list | FormalParameter list |
| `outputs` | list | FormalParameter list |
| `attributes` | dict | name → AttributeProto |
| `type_constraints` | list | TypeConstraintParam list |
| `doc` | str | Documentation string |

In [ ]:
# Programmatic schema queries

# Query 1: Find all operators that accept variadic inputs
variadic_ops = []
for schema in defs.get_all_schemas():
    if schema.domain in ('', 'ai.onnx'):
        for p in schema.inputs:
            opt = str(getattr(p, 'option', ''))
            if 'Variadic' in opt:
                variadic_ops.append((schema.name, p.name))
                break

print(f"Operators with variadic inputs ({len(variadic_ops)}):")
for op, param in sorted(variadic_ops)[:15]:
    print(f"  {op:<25} (variadic input: {param})")
if len(variadic_ops) > 15:
    print(f"  ... and {len(variadic_ops) - 15} more")

# Query 2: Find operators with optional inputs
optional_ops = []
for schema in defs.get_all_schemas():
    if schema.domain in ('', 'ai.onnx'):
        opt_inputs = [p.name for p in schema.inputs
                      if 'Optional' in str(getattr(p, 'option', ''))]
        if opt_inputs:
            optional_ops.append((schema.name, opt_inputs))

print(f"\nOperators with optional inputs ({len(optional_ops)}):")
for op, params in sorted(optional_ops)[:10]:
    print(f"  {op:<25} optional: {params}")

In [ ]:
# Query 3: Find operators introduced at each opset version
def when_introduced(op_type, domain=""):
    """Find the earliest opset where an operator was introduced."""
    all_schemas = defs.get_all_schemas_with_history()
    versions = [s.since_version for s in all_schemas
                if s.name == op_type and s.domain in (domain, 'ai.onnx' if domain == '' else domain)]
    return min(versions) if versions else None

# When were common ops introduced?
ops_of_interest = [
    "Relu", "Conv", "MatMul", "Gemm", "Softmax", "BatchNormalization",
    "LayerNormalization", "GroupNormalization", "Gelu", "Einsum",
    "ReduceSum", "Resize", "DynamicQuantizeLinear", "QuantizeLinear",
    "If", "Loop", "Scan", "HardSwish", "Mish", "Trilu",
]

print(f"{'Operator':<25} │ {'Introduced':>11} │ Timeline")
print("─" * 60)
for op in sorted(ops_of_interest):
    v = when_introduced(op)
    if v is not None:
        bar = ' ' * (v - 1) + '█' + '─' * (opset - v)
        print(f"{op:<25} │ {'opset ' + str(v):>11} │ {bar}")
    else:
        print(f"{op:<25} │ {'not found':>11} │")

# Query 4: Check if specific op exists at specific opset
print(f"\nOperator availability at specific opsets:")
check_pairs = [
    ("Relu", 7), ("Gelu", 15), ("Gelu", 20),
    ("LayerNormalization", 13), ("LayerNormalization", 17),
]
for op, ver in check_pairs:
    try:
        defs.get_schema(op, ver, '')
        print(f"  {op} at opset {ver}: Available")
    except Exception:
        print(f"  {op} at opset {ver}: NOT available")

<a id='6-reading-documentation'></a>
## 6. Reading Operator Documentation

A structured approach to reading and understanding operator documentation:

### Step-by-Step Reading Guide

```
  ┌───────────────────────────────────────────────────────────┐
  │           HOW TO READ AN OPERATOR SCHEMA                  │
  ├───────────────────────────────────────────────────────────┤
  │                                                           │
  │  Step 1: IDENTIFY                                         │
  │    ├── op_type, domain, since_version                     │
  │    └── Which opset version is your model using?           │
  │                                                           │
  │  Step 2: INPUTS                                           │
  │    ├── Count and order of inputs                          │
  │    ├── Which are optional vs required?                    │
  │    └── Map value names from your graph to formal params   │
  │                                                           │
  │  Step 3: OUTPUTS                                          │
  │    ├── Count and arity                                    │
  │    └── Shape relationship to inputs                       │
  │                                                           │
  │  Step 4: ATTRIBUTES                                       │
  │    ├── Which are required vs optional?                    │
  │    ├── Default values                                     │
  │    └── Enum-like constraints (auto_pad modes, etc.)       │
  │                                                           │
  │  Step 5: TYPE CONSTRAINTS                                 │
  │    ├── Allowed element types per type variable             │
  │    └── Cross-input type consistency requirements           │
  │                                                           │
  │  Step 6: SEMANTICS                                        │
  │    ├── Mathematical formula                               │
  │    ├── Broadcasting behavior                              │
  │    └── Edge cases and special values                      │
  │                                                           │
  └───────────────────────────────────────────────────────────┘
```

In [ ]:
# Structured schema reader
def read_schema_structured(op_type, opset_ver=opset, domain=""):
    """Read a schema in the structured 6-step approach."""
    try:
        s = defs.get_schema(op_type, opset_ver, domain)
    except Exception as e:
        print(f"Schema not found: {e}")
        return

    print(f"\n{'━' * 70}")
    print(f"  SCHEMA REPORT: {s.name}")
    print(f"{'━' * 70}")

    # Step 1: Identity
    print(f"\n  Step 1 — IDENTITY")
    print(f"    op_type:       {s.name}")
    print(f"    domain:        {domain if domain else '(default)'}")
    print(f"    since_version: {s.since_version}")
    print(f"    queried at:    opset {opset_ver}")

    # Step 2: Inputs
    print(f"\n  Step 2 — INPUTS ({len(s.inputs)})")
    for i, p in enumerate(s.inputs):
        opt = str(getattr(p, 'option', 'Single'))
        req = 'required' if 'Single' in opt else ('optional' if 'Optional' in opt else 'variadic')
        print(f"    [{i}] {p.name:<12} type={p.type_str:<5} arity={req}")

    # Step 3: Outputs
    print(f"\n  Step 3 — OUTPUTS ({len(s.outputs)})")
    for i, p in enumerate(s.outputs):
        print(f"    [{i}] {p.name:<12} type={p.type_str}")

    # Step 4: Attributes
    print(f"\n  Step 4 — ATTRIBUTES ({len(s.attributes)})")
    for name, attr in sorted(s.attributes.items()):
        req = 'REQUIRED' if attr.required else 'optional'
        type_name = attr_type_names.get(attr.type, f'type={attr.type}')
        print(f"    {name:<20} {type_name:<10} [{req}]")

    # Step 5: Type constraints
    print(f"\n  Step 5 — TYPE CONSTRAINTS ({len(s.type_constraints)})")
    for tc in s.type_constraints:
        types = list(tc.allowed_type_strs)
        short_types = [t.replace('tensor(', '').rstrip(')') for t in types[:8]]
        suffix = f' +{len(types)-8} more' if len(types) > 8 else ''
        print(f"    {tc.type_param_str}: {{{', '.join(short_types)}{suffix}}}")

    # Step 6: Semantics (doc excerpt)
    print(f"\n  Step 6 — SEMANTICS")
    doc = s.doc.strip() if s.doc else "(no documentation)"
    for line in doc.split('\n')[:8]:
        print(f"    {line.strip()[:80]}")
    if doc.count('\n') > 8:
        print(f"    ... ({doc.count(chr(10)) - 8} more lines)")

# Read a few schemas in structured format
read_schema_structured("Conv")
read_schema_structured("Gather")

<a id='7-attribute-versioning'></a>
## 7. Attribute Versioning

Attributes can **change across opset versions**. Understanding these changes is
critical for model portability and version conversion.

### Types of Attribute Changes

| Change Type | Example | Impact |
|-------------|---------|--------|
| **New attribute added** | ReduceSum.noop_with_empty_axes (opset 18) | Backward compatible |
| **Default changed** | Softmax.axis (opset 13: default=-1) | Semantic change |
| **Attribute → Input** | ReduceSum.axes (opset 13) | Breaking structural change |
| **Attribute removed** | BatchNorm.spatial (opset 9) | Must handle migration |
| **Type changed** | (rare) | Breaking change |

### Tracking Attribute Changes

```
  Softmax attribute history:
  ┌──────────┬────────────────────────────────────────────┐
  │ Opset 1  │ axis: int (default=1)                     │
  │          │ Semantics: coerce to 2D at axis, softmax   │
  │          │ over rightmost part                         │
  ├──────────┼────────────────────────────────────────────┤
  │ Opset 13 │ axis: int (default=-1)  ← CHANGED!        │
  │          │ Semantics: softmax along single axis        │
  │          │ This is a SEMANTIC + DEFAULT change         │
  └──────────┴────────────────────────────────────────────┘
```

In [ ]:
# Track attribute changes across opset versions
def track_attribute_changes(op_type, domain=""):
    """Track how attributes change across opset versions."""
    all_schemas = defs.get_all_schemas_with_history()
    versions = sorted(set(
        s.since_version for s in all_schemas
        if s.name == op_type and s.domain in (domain, 'ai.onnx' if domain == '' else domain)
    ))

    version_attrs = {}
    for v in versions:
        try:
            schema = defs.get_schema(op_type, max(v, 7), domain)
            attrs = set(schema.attributes.keys())
            version_attrs[v] = attrs
        except Exception:
            continue

    return version_attrs

# Track changes for operators known to have attribute modifications
for op in ["Softmax", "ReduceSum", "Squeeze", "BatchNormalization", "Resize", "Pad"]:
    print(f"\n{op} attribute history:")
    history = track_attribute_changes(op)
    all_attrs = set()
    for attrs in history.values():
        all_attrs |= attrs

    prev_attrs = set()
    for v, attrs in sorted(history.items()):
        added = attrs - prev_attrs
        removed = prev_attrs - attrs
        changes = []
        if added:
            changes.append(f"+{added}")
        if removed:
            changes.append(f"-{removed}")
        change_str = ' '.join(changes) if changes else '(initial)'
        print(f"  opset {v:>2}: {sorted(attrs)}")
        if changes:
            print(f"           changes: {change_str}")
        prev_attrs = attrs

# Compare Softmax attributes between opset 11 and 13
print(f"\nSoftmax attribute comparison:")
for ver in [11, 13, 17]:
    try:
        schema = defs.get_schema("Softmax", ver)
        print(f"  Opset {ver}: since_version={schema.since_version}, "
              f"attrs={sorted(schema.attributes.keys())}")
        for name, attr in schema.attributes.items():
            print(f"    {name}: required={attr.required}")
    except Exception as e:
        print(f"  Opset {ver}: {e}")

<a id='8-shape-inference'></a>
## 8. Shape Inference

Shape inference propagates tensor shapes through the graph **statically** (without
executing it). Each operator schema defines rules for computing output shapes from
input shapes and attributes.

### Formal Shape Propagation Rules

| Operator | Shape Rule | Formula |
|----------|------------|----------|
| Identity/Relu/BN | Pass-through | $\text{shape}(Y) = \text{shape}(X)$ |
| Add/Mul/Sub | Broadcast | $\text{shape}(Y) = \text{broadcast}(A, B)$ |
| MatMul | Contraction | $[\ldots, M, K] \times [\ldots, K, N] \to [\ldots, M, N]$ |
| Conv | Spatial formula | $d_{out} = \lfloor(d_{in} + 2p - d(k-1) - 1)/s + 1\rfloor$ |
| Reshape | Target shape | $\text{shape}(Y) = \text{shape\_input}$ (with -1 inference) |
| Transpose | Permutation | $\text{shape}(Y)[i] = \text{shape}(X)[\text{perm}[i]]$ |
| Concat | Join axis | $d_{\text{axis}} = \sum_i d_{\text{axis},i}$ |
| Flatten | Split at axis | $[d_1{\cdot}{\ldots}{\cdot}d_{\text{axis}}, d_{\text{axis}+1}{\cdot}{\ldots}{\cdot}d_n]$ |

### Shape Inference Cascade

```
  Input [batch, 3, 224, 224]    (known)
      │
      ▼ Conv(kernel=7, stride=2, pad=3)
  ┌──────────┐
  │ shape =  │  [(224+6-7)/2+1] = [112]
  │ [b,64,   │  → [batch, 64, 112, 112]
  │  112,112]│
  └──────────┘
      │
      ▼ MaxPool(kernel=3, stride=2, pad=1)
  ┌──────────┐
  │ shape =  │  [(112+2-3)/2+1] = [56]
  │ [b,64,   │  → [batch, 64, 56, 56]
  │  56, 56] │
  └──────────┘
      │
      ▼ Flatten(axis=1)
  ┌──────────┐
  │ shape =  │  [64×56×56] = [200704]
  │[b,200704]│  → [batch, 200704]
  └──────────┘
```

In [ ]:
# Demonstrate shape inference through a multi-layer model
np.random.seed(42)

# Build a ResNet-like block
X = helper.make_tensor_value_info("X", TensorProto.FLOAT, ["batch", 3, 224, 224])
Y = helper.make_tensor_value_info("Y", TensorProto.FLOAT, None)

inits = [
    numpy_helper.from_array(np.random.randn(64, 3, 7, 7).astype(np.float32)*0.01, "conv1.w"),
    numpy_helper.from_array(np.zeros(64, dtype=np.float32), "conv1.b"),
    numpy_helper.from_array(np.ones(64, dtype=np.float32), "bn1.scale"),
    numpy_helper.from_array(np.zeros(64, dtype=np.float32), "bn1.bias"),
    numpy_helper.from_array(np.zeros(64, dtype=np.float32), "bn1.mean"),
    numpy_helper.from_array(np.ones(64, dtype=np.float32), "bn1.var"),
    numpy_helper.from_array(np.random.randn(64, 64, 3, 3).astype(np.float32)*0.01, "conv2.w"),
]

nodes = [
    helper.make_node("Conv", ["X", "conv1.w", "conv1.b"], ["conv1"],
                     kernel_shape=[7, 7], strides=[2, 2], pads=[3, 3, 3, 3]),
    helper.make_node("BatchNormalization",
                     ["conv1", "bn1.scale", "bn1.bias", "bn1.mean", "bn1.var"],
                     ["bn1"]),
    helper.make_node("Relu", ["bn1"], ["relu1"]),
    helper.make_node("MaxPool", ["relu1"], ["pool1"],
                     kernel_shape=[3, 3], strides=[2, 2], pads=[1, 1, 1, 1]),
    helper.make_node("Conv", ["pool1", "conv2.w"], ["conv2"],
                     kernel_shape=[3, 3], pads=[1, 1, 1, 1]),
    helper.make_node("Relu", ["conv2"], ["Y"]),
]

graph = helper.make_graph(nodes, "resnet_block", [X], [Y], initializer=inits)
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 17)])

# Run shape inference
model_inferred = shape_inference.infer_shapes(model)

def get_shape(tensor_type):
    if not tensor_type.HasField('shape'):
        return 'unknown'
    return [d.dim_param if d.dim_param else d.dim_value
            for d in tensor_type.shape.dim]

print(f"Shape inference cascade:")
print(f"{'Tensor':<15} │ {'Op':>20} │ {'Inferred Shape':>30}")
print("─" * 70)

# Input
for inp in model_inferred.graph.input:
    shape = get_shape(inp.type.tensor_type)
    print(f"{inp.name:<15} │ {'(input)':>20} │ {str(shape):>30}")

# Intermediate values
node_map = {n.output[0]: n.op_type for n in model_inferred.graph.node}
for vi in model_inferred.graph.value_info:
    shape = get_shape(vi.type.tensor_type)
    op = node_map.get(vi.name, '?')
    print(f"{vi.name:<15} │ {op:>20} │ {str(shape):>30}")

# Output
for out in model_inferred.graph.output:
    shape = get_shape(out.type.tensor_type)
    op = node_map.get(out.name, '?')
    print(f"{out.name:<15} │ {op + ' (output)':>20} │ {str(shape):>30}")

In [ ]:
# Manual shape inference verification
def conv_output_shape(h_in, w_in, k, s, p_h, p_w, d=1):
    h_out = (h_in + 2*p_h - d*(k-1) - 1) // s + 1
    w_out = (w_in + 2*p_w - d*(k-1) - 1) // s + 1
    return h_out, w_out

def pool_output_shape(h_in, w_in, k, s, p_h, p_w):
    return conv_output_shape(h_in, w_in, k, s, p_h, p_w)

# Trace through the model manually
h, w = 224, 224
print(f"Manual shape inference:")
print(f"  Input:    [batch, 3, {h}, {w}]")

h, w = conv_output_shape(h, w, k=7, s=2, p_h=3, p_w=3)
print(f"  Conv1:    [batch, 64, {h}, {w}]   (k=7, s=2, p=3)")

print(f"  BN1:      [batch, 64, {h}, {w}]   (identity shape)")
print(f"  Relu1:    [batch, 64, {h}, {w}]   (identity shape)")

h, w = pool_output_shape(h, w, k=3, s=2, p_h=1, p_w=1)
print(f"  MaxPool:  [batch, 64, {h}, {w}]   (k=3, s=2, p=1)")

h, w = conv_output_shape(h, w, k=3, s=1, p_h=1, p_w=1)
print(f"  Conv2:    [batch, 64, {h}, {w}]   (k=3, s=1, p=1)")

print(f"  Relu2:    [batch, 64, {h}, {w}]   (identity shape)")
print(f"  Flatten:  [batch, {64 * h * w}]")

<a id='9-complete-schema-audit'></a>
## 9. Complete Schema Audit

A schema audit systematically validates every node in a model against its
operator schema. This is essential for debugging and deployment validation.

### Audit Checklist

For each node in the graph:

1. **Schema exists** — Does the (op_type, domain, opset) combination have a schema?
2. **Input count** — Does the node have the right number of inputs?
3. **Output count** — Does the node have the right number of outputs?
4. **Required attributes** — Are all required attributes present?
5. **Type consistency** — Do all inputs sharing a type variable have the same type?
6. **Shape inference** — Can shapes be inferred for all outputs?

In [ ]:
# Complete schema audit tool
def audit_model_schemas(model):
    """Perform a complete schema audit of an ONNX model."""
    report = {
        'total_nodes': len(model.graph.node),
        'findings': [],
        'stats': {
            'standard_ops': 0,
            'custom_ops': 0,
            'unique_ops': set(),
            'domains': set(),
        }
    }

    imported_opsets = {}
    for oi in model.opset_import:
        imported_opsets[oi.domain] = oi.version

    for i, node in enumerate(model.graph.node):
        domain = node.domain if node.domain else ''
        report['stats']['unique_ops'].add(node.op_type)
        report['stats']['domains'].add(domain if domain else '(default)')

        if domain in ('', 'ai.onnx'):
            report['stats']['standard_ops'] += 1
            opset_ver = imported_opsets.get('', imported_opsets.get('ai.onnx', 17))

            try:
                schema = defs.get_schema(node.op_type, opset_ver, '')

                # Check input count
                min_inputs = sum(1 for p in schema.inputs
                                 if 'Single' in str(getattr(p, 'option', 'Single')))
                actual_inputs = len([x for x in node.input if x])
                if actual_inputs < min_inputs:
                    report['findings'].append({
                        'node': i,
                        'op': node.op_type,
                        'severity': 'ERROR',
                        'message': f'Too few inputs: {actual_inputs} < {min_inputs} required',
                    })

                # Check required attributes
                node_attrs = {a.name for a in node.attribute}
                for attr_name, attr in schema.attributes.items():
                    if attr.required and attr_name not in node_attrs:
                        report['findings'].append({
                            'node': i,
                            'op': node.op_type,
                            'severity': 'ERROR',
                            'message': f'Missing required attribute: {attr_name}',
                        })

            except Exception as e:
                report['findings'].append({
                    'node': i,
                    'op': node.op_type,
                    'severity': 'ERROR',
                    'message': f'Schema not found: {e}',
                })
        else:
            report['stats']['custom_ops'] += 1
            report['findings'].append({
                'node': i,
                'op': node.op_type,
                'severity': 'INFO',
                'message': f'Custom domain: {domain}',
            })

    return report

# Audit our ResNet-like model
audit = audit_model_schemas(model_inferred)

print(f"Schema Audit Report")
print(f"{'=' * 60}")
print(f"  Total nodes:    {audit['total_nodes']}")
print(f"  Standard ops:   {audit['stats']['standard_ops']}")
print(f"  Custom ops:     {audit['stats']['custom_ops']}")
print(f"  Unique ops:     {sorted(audit['stats']['unique_ops'])}")
print(f"  Domains:        {sorted(audit['stats']['domains'])}")

if audit['findings']:
    print(f"\n  Findings ({len(audit['findings'])})")
    for f in audit['findings']:
        print(f"    [{f['severity']}] Node {f['node']} ({f['op']}): {f['message']}")
else:
    print(f"\n  No issues found — all schemas validated successfully.")

# Also verify with checker
try:
    checker.check_model(model_inferred)
    print(f"\n  onnx.checker.check_model: PASSED")
except Exception as e:
    print(f"\n  onnx.checker.check_model: FAILED ({e})")

In [ ]:
# Visualize operator usage patterns
import matplotlib.pyplot as plt

# Count operators across all default domain schemas by category
category_map = {
    'Elementwise': ['Add', 'Sub', 'Mul', 'Div', 'Neg', 'Abs', 'Exp', 'Log', 'Sqrt', 'Pow'],
    'Linear Algebra': ['MatMul', 'Gemm', 'Einsum'],
    'Activation': ['Relu', 'Sigmoid', 'Tanh', 'Softmax', 'Gelu', 'LeakyRelu'],
    'Conv/Pool': ['Conv', 'ConvTranspose', 'MaxPool', 'AveragePool', 'GlobalAveragePool'],
    'Normalization': ['BatchNormalization', 'LayerNormalization', 'InstanceNormalization'],
    'Shape': ['Reshape', 'Transpose', 'Squeeze', 'Unsqueeze', 'Flatten'],
    'Reduction': ['ReduceMean', 'ReduceSum', 'ReduceMax', 'ArgMax'],
    'Data Movement': ['Gather', 'Concat', 'Split', 'Slice', 'Pad'],
}

# Count type constraint diversity
type_diversity = {}
for schema in defs.get_all_schemas():
    if schema.domain in ('', 'ai.onnx'):
        total_types = sum(len(tc.allowed_type_strs) for tc in schema.type_constraints)
        type_diversity[schema.name] = total_types

# Most type-flexible operators
top_flexible = sorted(type_diversity.items(), key=lambda x: -x[1])[:15]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Category sizes
cat_names = list(category_map.keys())
cat_sizes = [len(v) for v in category_map.values()]
colors = plt.cm.Set2(np.linspace(0, 1, len(cat_names)))
ax1.barh(cat_names, cat_sizes, color=colors, edgecolor='white')
ax1.set_xlabel('Number of Operators (sample)')
ax1.set_title('Operator Categories', fontweight='bold')
ax1.invert_yaxis()
for i, v in enumerate(cat_sizes):
    ax1.text(v + 0.1, i, str(v), va='center')

# Type flexibility
names = [x[0] for x in top_flexible]
counts = [x[1] for x in top_flexible]
ax2.barh(names, counts, color='steelblue', edgecolor='navy', alpha=0.8)
ax2.set_xlabel('Total Allowed Types')
ax2.set_title('Most Type-Flexible Operators', fontweight='bold')
ax2.invert_yaxis()

plt.tight_layout()
plt.show()

<a id='10-key-takeaways'></a>
## 10. Key Takeaways

1. **Operator schemas** are the authoritative contracts that define ONNX operator behavior.
   They specify inputs, outputs, attributes, type constraints, and semantics.

2. **Type constraints** use type variables (e.g., $T$) with allowed type sets:
   $$T \in \{\text{float16}, \text{float32}, \text{float64}\}$$
   All inputs sharing a type variable must have the same element type.

3. **Broadcasting** follows numpy rules: dimensions are compatible if equal or one is 1.
   Shapes are right-aligned; the output dimension is $\max(d_A, d_B)$.
   ONNX supports both **multidirectional** (Add, Mul) and **unidirectional** (Gemm bias) broadcasting.

4. **`onnx.defs`** is your primary tool for programmatic schema inspection:
   `get_schema()`, `get_all_schemas()`, `get_all_schemas_with_history()`.

5. **Attributes are versioned** — they can be added, have defaults changed, or be
   promoted from attributes to inputs across opset versions (e.g., ReduceSum axes at opset 13).

6. **Shape inference** propagates tensor shapes through the graph using operator-specific
   rules. Each operator has a formal shape rule (identity, broadcast, contraction, spatial).

7. A **structured reading approach** (6 steps: identity, inputs, outputs, attributes,
   type constraints, semantics) transforms schema inspection from guesswork into
   systematic deduction.

8. Regular **schema audits** validate that every node in a model conforms to its
   operator schema — checking input counts, required attributes, type consistency,
   and shape propagation.